# YOLOv8 Alan Tespit Modeli — Colab Eğitimi

Sınav kağıdında `not` ve `ogrenci_numara` alanlarını bulan detektörü Colab GPU'sunda eğitir.

**Başlamadan önce:**
1. Menüden **Çalışma Zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin.
2. `roboflow_export.zip` dosyasını Google Drive'ınızın ana klasörüne yükleyin
   (yerelde `data/` klasöründen `zip -r roboflow_export.zip roboflow_export` ile üretilir).
3. Hücreleri sırayla çalıştırın. Eğitim bitince `yolo_fields.pt` otomatik iner —
   onu projede `models/yolo_fields.pt` olarak kaydedin.

In [ ]:
# GPU kontrolü — 'Tesla T4' benzeri bir kart görünmeli
!nvidia-smi -L

In [ ]:
!pip -q install ultralytics

In [ ]:
# Drive'ı bağla ve veri setini aç (zip Drive ana klasöründe olmalı)
from google.colab import drive
drive.mount('/content/drive')

!unzip -q /content/drive/MyDrive/roboflow_export.zip -d /content/
!ls /content/roboflow_export

In [ ]:
# data.yaml yollarını Colab konumuna göre yaz
import yaml

yaml_path = '/content/roboflow_export/data.yaml'
with open(yaml_path) as f:
    data = yaml.safe_load(f)

data['path'] = '/content/roboflow_export'
data['train'] = 'train/images'
data['val'] = 'valid/images'
data['test'] = 'test/images'

with open(yaml_path, 'w') as f:
    yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)

print(open(yaml_path).read())

In [ ]:
# Eğitim — training/train_yolo.py ile aynı ayarlar (batch T4 için 16'ya çıkarıldı)
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

model.train(
    data='/content/roboflow_export/data.yaml',
    epochs=300,
    imgsz=960,
    batch=16,
    device=0,
    project='/content/runs',
    name='fields',
    patience=25,
    # Şablon ezberini kırmaya yönelik augmentation
    degrees=5.0,
    translate=0.15,
    scale=0.6,
    perspective=0.0005,
)

In [ ]:
# Test split üzerinde değerlendirme
best = YOLO('/content/runs/fields/weights/best.pt')
metrics = best.val(data='/content/roboflow_export/data.yaml', split='test', device=0)
print(f"mAP50: {metrics.box.map50:.3f} | mAP50-95: {metrics.box.map:.3f}")

In [ ]:
# En iyi modeli indir — projede models/yolo_fields.pt olarak kaydedin.
# (Yedek olarak Drive'a da kopyalanır: MyDrive/yolo_fields.pt)
import shutil
shutil.copy('/content/runs/fields/weights/best.pt', '/content/drive/MyDrive/yolo_fields.pt')

from google.colab import files
shutil.copy('/content/runs/fields/weights/best.pt', '/content/yolo_fields.pt')
files.download('/content/yolo_fields.pt')